In [1]:
import vents_grib as vg
import vents as v
import polaires

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
vg.df

,latitude,longitude,valid_time,force,direction,temps,x_data,y_data
0,31.25,-32.75,2026-03-04 06:00:00,6.557166,6.483459,0.0,-1375.5,1875.0
1,31.25,-32.50,2026-03-04 06:00:00,6.759312,6.952271,0.0,-1365.0,1875.0
2,31.25,-32.25,2026-03-04 06:00:00,7.053451,6.979065,0.0,-1354.5,1875.0
3,31.25,-32.00,2026-03-04 06:00:00,7.437273,6.466507,0.0,-1344.0,1875.0
4,31.25,-31.75,2026-03-04 06:00:00,7.699679,5.663498,0.0,-1333.5,1875.0
...,...,...,...,...,...,...,...,...
1437826,55.25,11.75,2026-03-14 06:00:00,12.080581,129.060349,240.0,493.5,3315.0
1437827,55.25,12.00,2026-03-14 06:00:00,12.347339,135.959961,240.0,504.0,3315.0
1437828,55.25,12.25,2026-03-14 06:00:00,14.841729,136.913239,240.0,514.5,3315.0
1437829,55.25,12.50,2026-03-14 06:00:00,17.887060,132.579941,240.0,525.0,3315.0


In [3]:
VENT = vg.gfs
POLAIRE = polaires.Vector_vect

In [4]:
%matplotlib qt

In [5]:
dt = 0.25
n = 100

14.8 * dt * np.pi / n

0.11623892818282235

In [6]:
import cProfile
import isochrone_partielle_angle as iso

In [12]:
# p_dep = [46, 2]
# p_arr = [44, 9]
p_dep = [49, -14]
p_arr = [44, -2]
t = 170

routage = iso.routage(p_dep, p_arr, t, dt = 1, n = 100, V = VENT, P = POLAIRE, e_arr = 5, r = 1.5, ang = np.pi/2, dang = np.radians(0.9), delta = 1)

# cProfile.run("routage = iso.routage(p_dep, p_arr, t, dt = 1, n = 100, V = VENT, P = POLAIRE, e_arr = 5, r = 1.5, ang = np.pi/2, dang = np.radians(1.2), delta = 1)",
#             sort="tottime")


nombre isochrones : 0, temps : 170.00 heures, nombre de points : 1
nombre isochrones : 1, temps : 171.00 heures, nombre de points : 100
nombre isochrones : 2, temps : 172.00 heures, nombre de points : 89
nombre isochrones : 3, temps : 173.00 heures, nombre de points : 129
nombre isochrones : 4, temps : 174.00 heures, nombre de points : 174
nombre isochrones : 5, temps : 175.00 heures, nombre de points : 213
nombre isochrones : 6, temps : 176.00 heures, nombre de points : 254
nombre isochrones : 7, temps : 177.00 heures, nombre de points : 295
nombre isochrones : 8, temps : 178.00 heures, nombre de points : 331
nombre isochrones : 9, temps : 179.00 heures, nombre de points : 369
nombre isochrones : 10, temps : 180.00 heures, nombre de points : 398
nombre isochrones : 11, temps : 181.00 heures, nombre de points : 437
nombre isochrones : 12, temps : 182.00 heures, nombre de points : 473
nombre isochrones : 13, temps : 183.00 heures, nombre de points : 496
nombre isochrones : 14, temps : 1

In [13]:
latitude, longitude, time_list, L = routage

In [14]:
t_arr = time_list[-1]
print((t_arr - t) // 24, "jours", (t_arr - t) % 24, "heures")

2 jours 10 heures


In [15]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())

ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)

gl = ax.gridlines(draw_labels=True,
                  crs=ccrs.PlateCarree(),
                  linestyle='--',
                  alpha=0.6)

gl.top_labels = False
gl.right_labels = False

# -------------------------
# GRILLE
# -------------------------
V = VENT
t = 6

intx = [350, 360]
inty = [42, 49]
n = 60

lon = np.linspace(intx[0], intx[1], n+1)
lat = np.linspace(inty[0], inty[1], n+1)

X, Y = np.meshgrid(lon, lat)

# -------------------------
# CALCUL DU VENT
# -------------------------
U = np.zeros_like(X)
Vv = np.zeros_like(Y)
C = np.zeros_like(X)

for i in range(n+1):
    for j in range(n+1):
        direction, speed = V([X[i,j], Y[i,j]], t, ref = 'deg')
        
        U[i,j] = - speed * np.sin(np.radians(direction))
        Vv[i,j] = - speed * np.cos(np.radians(direction))
        C[i,j] = speed

# -------------------------
# QUIVER
# -------------------------
q = ax.quiver(X, Y, U, Vv, C,
              transform=ccrs.PlateCarree(),
              cmap='Greys')

ax.set_extent([intx[0], intx[1], inty[0], inty[1]],
              crs=ccrs.PlateCarree())

# route 
ax.plot(longitude, latitude, transform=ccrs.PlateCarree(), c = 'red')
# ax.scatter(longitude, latitude, transform=ccrs.PlateCarree(), c = 'red')

plt.show()

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import datetime, timedelta

# -------------------------
# Paramètres de la grille
# -------------------------
V = VENT  # votre fonction de vent
intx = [min(pd.unique(vg.df['longitude'])), max(pd.unique(vg.df['longitude']))]
inty = [min(pd.unique(vg.df['latitude'])), max(pd.unique(vg.df['latitude']))]
intt = [0, max(pd.unique(vg.df['temps']))]
n = 50

start_date = vg.df['valid_time'][0]

lon = np.linspace(intx[0], intx[1], n+1)
lat = np.linspace(inty[0], inty[1], n+1)
X, Y = np.meshgrid(lon, lat)

# -------------------------
# Initialisation figure
# -------------------------
fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=ccrs.Mercator())
ax.add_feature(cfeature.BORDERS, linewidth=1)
ax.add_feature(cfeature.COASTLINE, linewidth=1)

gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(),
                  linestyle='--', alpha=0.6)
gl.top_labels = False
gl.right_labels = False

ax.set_extent([intx[0], intx[1], inty[0], inty[1]],
              crs=ccrs.PlateCarree())

# -------------------------
# Fonction pour calculer vent
# -------------------------
def compute_wind(t):
    U = np.zeros_like(X)
    Vv = np.zeros_like(Y)
    C = np.zeros_like(X)
    for i in range(n+1):
        for j in range(n+1):
            direction, speed = V([X[i,j], Y[i,j]], t, ref='deg')
            U[i,j] = - speed * np.sin(np.radians(direction))
            Vv[i,j] = - speed * np.cos(np.radians(direction))
            C[i,j] = speed
    return U, Vv, C

# -------------------------
# Vent initial t=0
# -------------------------
initial_time = intt[0]
U, Vv, C = compute_wind(initial_time)
q = ax.quiver(X, Y, U, Vv, C, transform=ccrs.PlateCarree(), cmap='Greys')


# -------------------------
# Route
# -------------------------
for i in range(len(latitude)):
    l = L[L[:, 2] == i]
    ax.plot(l[:, 1], l[:, 0], transform=ccrs.PlateCarree(), c = 'lightcoral')
    
route_line, = ax.plot(longitude, latitude, transform=ccrs.PlateCarree(), c='red')
boat_scatter = ax.scatter(latitude[0], longitude[0], c='red', transform=ccrs.PlateCarree())


# -------------------------
# Slider
# -------------------------
ax_slider = plt.axes([0.2, 0.02, 0.6, 0.03])
slider = Slider(ax_slider, 'Temps', valmin=intt[0], valmax=intt[1], valinit=intt[0], valstep=1, facecolor='grey')
slider.vline.set_visible(False)

initial_date = start_date + timedelta(hours=initial_time)
slider.valtext.set_text(initial_date.strftime("%d/%m %H:%M"))

def update(val):
    t = slider.val
    U, Vv, C = compute_wind(t)
    q.set_UVC(U, Vv, C)

    # date
    current_date = start_date + timedelta(hours=t)
    slider.valtext.set_text(current_date.strftime("%d/%m %H:%M"))

    # Route
    idx = np.argmin(np.abs(time_list - t))
    boat_scatter.set_offsets([[longitude[idx], latitude[idx]]])


    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()